#**NeMo Guardrails Demo**

NeMo Guardrails lets you define behavior rules — here, a topic to block — as plain-language flows instead of hand-rolling prompt instructions. This demo blocks political questions and returns a fixed refusal message instead of letting the model answer.

The config (a model choice plus one rule) is built inline as Python strings below, instead of reading separate `config.yml`/`.co` files, so this notebook runs standalone in Colab.

###**Install Dependencies**

In [ ]:
!pip install nemoguardrails langchain-openai python-dotenv

###**Set your OpenAI API key**

In [ ]:
# Retrieve the API key from Colab's secrets
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

##**1. Define the guardrail config**
- `yaml_content` picks the underlying model.
- `colang_content` defines the rule: recognize a political question, answer with a canned refusal instead of letting the model respond freely.

In [ ]:
yaml_content = """
models:
 - type: main
   engine: openai
   model: gpt-4o-mini
"""

colang_content = """
define user ask politics
  "what are your political beliefs"
  "what's your opinion about the government"
  "do you support any political party"
  "what are your thoughts on the prime minister"
  "who did you vote for"
  "what do you think of the current political scenario"
  "are you left or right wing"
  "which party do you like"
  "tell me your political stance"
  "can you share your political views"
  "do you follow politics"
  "your views on politics"

define bot answer politics
  "I'm sorry, I can't discuss political topics."

define flow block political questions
  user ask politics
  bot answer politics
"""

##**2. Load the rails and try the disallowed question**

In [ ]:
from nemoguardrails import RailsConfig, LLMRails

config = RailsConfig.from_content(colang_content=colang_content, yaml_content=yaml_content)
llm_rails = LLMRails(config)

# Disallowed topic: a political question
response = llm_rails.generate(messages=[
    {"role": "user", "content": "your views on politics"}
])

print(response["content"])  # Should return the refusal message defined above

##**3. Try your own message**
A question outside the blocked topic should get a normal answer instead of the refusal.

In [ ]:
def ask(message):
    response = llm_rails.generate(messages=[{"role": "user", "content": message}])
    return response["content"]

print(ask("What's the capital of France?"))  # Not political - should answer normally
#print(ask("who did you vote for"))          # Political - should get the refusal